In [1]:
# Analyze new long-format claims
import pandas as pd
from pathlib import Path

# Input file
claims_path = Path("../claims_raw.csv")

# Helper to load if exists
def load_csv(path: Path) -> pd.DataFrame | None:
    if path.exists():
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"Failed to read {path}: {e}")
    else:
        print(f"Warning: {path} not found")
    return None

claims_df = load_csv(claims_path)

# Summary: claims long format
if claims_df is not None:
    total = len(claims_df)
    non_empty = int(claims_df['claim'].fillna('').ne('').sum()) if 'claim' in claims_df.columns else 0
    unique_cases = claims_df['name'].nunique() if 'name' in claims_df.columns else None
    print(f"Claims: rows={total}, non_empty_claims={non_empty}, unique_cases={unique_cases}")

    # Show examples: first 5 claims for Supreme Court cases
    def is_supreme_case(row):
        court = str(row.get('court', '')).lower()
        name = str(row.get('name', '')).lower()
        docket = str(row.get('docket', '')).lower()
        return ('supreme' in court) or ('supreme court' in name) or ('scotus' in court) or ('supreme' in docket)

    # If claims csv lacks court, we can’t filter by court; fallback to first 5
    subset = claims_df
    if {'court','name','docket'}.issubset(set(claims_df.columns)):
        subset = claims_df[claims_df.apply(is_supreme_case, axis=1)]
    sample = subset.head(5)
    print(f"\nFirst 5 claims (Supreme Court filtered when possible)")
    display(sample.fillna(''))

Claims: rows=10480, non_empty_claims=10480, unique_cases=3226

First 5 claims (Supreme Court filtered when possible)


,name,docket,claim
0,Roe v. Wade,70-18,The Constitution protects a person's right to ...
1,Roe v. Wade,70-18,States cannot restrict abortion during the fir...
2,Roe v. Wade,70-18,"During the second three months of pregnancy, s..."
3,Roe v. Wade,70-18,"After the fetus can survive outside the womb, ..."
4,Stanley v. Illinois,70-5014,States cannot take children from unwed fathers...


In [3]:
# Check for exactly identical claims
if claims_df is not None and 'claim' in claims_df.columns:
    # Check for duplicates in the 'claim' column
    duplicates = claims_df[claims_df.duplicated(subset=['claim'], keep=False)]
    num_duplicates = len(duplicates)
    unique_duplicates = duplicates['claim'].nunique()
    
    print(f"Found {num_duplicates} rows with identical claims (across {unique_duplicates} unique claim strings).")
    
    if num_duplicates > 0:
        print("\n--- List of Identical Claims and their Sources ---")
        # Sort by claim to group them visually
        sorted_dupes = duplicates.sort_values('claim')
        
        # Group by claim and print details
        for claim_text, group in sorted_dupes.groupby('claim'):
            print(f"\n[Duplicate Claim]: \"{claim_text}\"")
            print(f"Found in {len(group)} entries:")
            # Show name and docket if available
            cols = [c for c in ['name', 'docket'] if c in group.columns]
            if cols:
                for _, row in group.iterrows():
                    source_info = ", ".join([f"{c}: {row[c]}" for c in cols])
                    print(f"  - {source_info}")
            else:
                # Fallback if metadata columns missing
                print(group.to_string(index=False))
else:
    print("Claims dataframe not loaded or missing 'claim' column.")

Found 16 rows with identical claims (across 7 unique claim strings).

--- List of Identical Claims and their Sources ---

[Duplicate Claim]: "A person cannot claim entrapment if they were already willing to commit the crime before law enforcement got involved."
Found in 2 entries:
  - name: Hampton v. United States, docket: 74-5822
  - name: Osborn v. United States, docket: 29

[Duplicate Claim]: "Employers cannot fire someone for being transgender."
Found in 2 entries:
  - name: R.G. & G.R. Harris Funeral Homes Inc. v. Equal Employment Opportunity Commission, docket: 18-107
  - name: R.G. & G.R. Harris Funeral Homes Inc. v. Equal Employment Opportunity Commission, docket: 18-107

[Duplicate Claim]: "Prosecutors cannot remove potential jurors from a jury because of their race."
Found in 4 entries:
  - name: Powers v. Ohio, docket: 89-5011
  - name: Miller-El v. Dretke, docket: 03-9659
  - name: Snyder v. Louisiana, docket: 06-10119
  - name: Flowers v. Mississippi, docket: 17-9572

[Du

In [4]:
# Drop duplicate claims and save
if claims_df is not None and 'claim' in claims_df.columns:
    initial_count = len(claims_df)
    # Drop duplicates, keeping the first occurrence
    claims_df_deduped = claims_df.drop_duplicates(subset=['claim'], keep='first')
    final_count = len(claims_df_deduped)
    dropped_count = initial_count - final_count
    
    print(f"Initial rows: {initial_count}")
    print(f"Rows after deduplication: {final_count}")
    print(f"Dropped {dropped_count} duplicate rows.")
    
    # Update the dataset
    try:
        claims_df_deduped.to_csv(claims_path, index=False)
        print(f"Successfully saved deduplicated dataset to {claims_path}")
        # Update the variable in memory
        claims_df = claims_df_deduped
    except Exception as e:
        print(f"Error saving file: {e}")
else:
    print("Claims dataframe not loaded or missing 'claim' column.")

Initial rows: 10480
Rows after deduplication: 10471
Dropped 9 duplicate rows.
Successfully saved deduplicated dataset to ../claims_raw.csv
